# Chapter 3 — Gender and Market Segmentation: Visualisation Draft

This notebook renders the planned 19 tables and 6 figures for Chapter 3 from the validated exports produced by Notebook 18. Run Notebook 18 through its export cell first.

This is a first visual draft. The existing Notebook 14 style is reused provisionally; table design will be reviewed before a global redesign is applied to the whole book.

**Methodological conventions:** dominant accord = position 1; dyad = positions 1+2; triad = positions 1+2+3, preserving source order. One profile per fragrance at each applicable level. Minimum support = 50 fragrances per classification–profile. Median segmentation is independent within each classification and analytical level. Votes are platform activity, not sales or unique consumers; ratings are platform evaluations, not objective quality measures.

## 0. Setup, load validated exports and prepare all tables

In [ ]:
%pip install great-tables

In [ ]:
# ============================================================
# CHAPTER 3 VISUALISATION DRAFT — SETUP AND DATA
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from great_tables import GT, style, loc

DATA_DIR = Path("/Volumes/main/fragrance_analytics/reference_data/chapter_3")
OUTPUT_DIR = Path("/Volumes/main/fragrance_analytics/reference_data/visualizations")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENDERS = ["Men", "Women", "Unisex"]
LEVELS = ["Dominant accords", "Dyads", "Triads"]
SEGMENTS = ["Standout Profiles", "Hidden Favourites", "Widely Explored", "Quiet Profiles"]

STYLE = {
    "bg":"#F7F3EC", "text":"#302824", "muted":"#756B64",
    "accent":"#A47C59", "grid":"#E9E2D8",
    "Men":"#687F8C", "Women":"#B47C83", "Unisex":"#A58B68"
}
SEGMENT_COLORS = {
    "Standout Profiles":"#A47C59", "Hidden Favourites":"#687F8C",
    "Widely Explored":"#B47C83", "Quiet Profiles":"#A58B68"
}
plt.rcParams.update({
    "figure.facecolor":STYLE["bg"], "axes.facecolor":STYLE["bg"],
    "savefig.facecolor":STYLE["bg"], "font.family":"DejaVu Sans",
    "text.color":STYLE["text"], "axes.labelcolor":STYLE["muted"],
    "xtick.color":STYLE["text"], "ytick.color":STYLE["text"]
})

def load_csv(name):
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing. Run Notebook 18 through its export cell.")
    return pd.read_csv(path)

gender_dist = load_csv("chapter_3_gender_distribution.csv")
prevalence = load_csv("chapter_3_accord_prevalence.csv")
thresholds = load_csv("chapter_3_gender_thresholds.csv")
segment_summary = load_csv("chapter_3_gender_segment_summary.csv")
dominant = load_csv("chapter_3_dominant_accord_profiles.csv")
dyads = load_csv("chapter_3_dyad_profiles.csv")
triads = load_csv("chapter_3_triad_profiles.csv")
examples = load_csv("chapter_3_top_three_fragrance_examples.csv")
level_comparison = load_csv("chapter_3_gender_level_comparison.csv")
segment_comparison = load_csv("chapter_3_gender_segment_comparison.csv")

PROFILE_DATA = {"Dominant accords":dominant, "Dyads":dyads, "Triads":triads}
freq_parts = []
for level, pdf in PROFILE_DATA.items():
    entity = {"Dominant accords":"dominant_accord","Dyads":"dyad","Triads":"triad"}[level]
    part = pdf[["gender_clean",entity,"fragrance_count"]].copy()
    part.columns = ["classification","profile","profile_frequency"]
    part["analytical_level"] = level
    freq_parts.append(part)
profile_freq = pd.concat(freq_parts, ignore_index=True)

examples["profile"] = examples["dominant_accord"].fillna(examples["dyad"]).fillna(examples["triad"])
examples = examples.merge(profile_freq, on=["analytical_level","classification","profile"], how="left")

assert len(gender_dist) == 3 and set(gender_dist.gender_clean) == set(GENDERS)
assert set(thresholds.minimum_fragrance_support.astype(int)) == {50}
assert (len(dominant),len(dyads),len(triads)) == (95,276,104)
assert len(examples) == 1425 and examples.profile_frequency.notna().all()

def make_table(df,title,subtitle):
    t = (GT(df).tab_header(title=title,subtitle=subtitle).tab_options(
        table_background_color=STYLE["bg"], heading_background_color=STYLE["bg"],
        column_labels_background_color=STYLE["bg"],
        table_font_names=["DejaVu Sans","Arial","sans-serif"],
        table_font_size="18px", table_font_color=STYLE["text"],
        heading_align="left", heading_title_font_size="24px",
        heading_subtitle_font_size="19px", heading_padding="12px",
        column_labels_font_weight="bold", column_labels_padding="9px",
        table_body_hlines_color=STYLE["grid"], table_body_hlines_width="1px",
        data_row_padding="9px"))
    t=t.tab_style(style.fill(color=STYLE["bg"]),locations=[loc.body(),loc.column_header(),loc.header()])
    t=t.tab_style(style.text(font="DejaVu Serif",size="24px",weight="bold",color=STYLE["text"]),locations=[loc.title()])
    t=t.tab_style(style.text(font="DejaVu Sans",size="19px",color=STYLE["muted"]),locations=[loc.subtitle()])
    return t

def show_table(df,title,subtitle):
    displayHTML('<div class="nodarkmode">'+make_table(df,title,subtitle).as_raw_html(all_important=True)+'</div>')

# Table 3.1
t31=gender_dist.rename(columns={"gender_clean":"Classification","fragrance_count":"Fragrance count","catalogue_share_pct":"Catalogue share (%)"})[["Classification","Fragrance count","Catalogue share (%)"]]

# Table 3.2
t32=(prevalence.sort_values(["gender_clean","prevalence_pct","accord"],ascending=[True,False,True])
     .groupby("gender_clean",group_keys=False).head(5)
     .rename(columns={"gender_clean":"Classification","accord":"Accord","fragrance_count":"Fragrance count","prevalence_pct":"Prevalence (%)"})
     [["Classification","Accord","Fragrance count","Prevalence (%)"]])

# Table 3.3
wide=prevalence.pivot(index="accord",columns="gender_clean",values="prevalence_pct").reset_index()
wide["Maximum spread (pp)"]=wide[GENDERS].max(axis=1)-wide[GENDERS].min(axis=1)
t33=wide.nlargest(10,"Maximum spread (pp)").rename(columns={"accord":"Accord"})[["Accord","Men","Women","Unisex","Maximum spread (pp)"]]

def thresholds_table(level):
    return thresholds[thresholds.analytical_level==level].rename(columns={
        "classification":"Classification","eligible_profile_count":"Eligible profiles",
        "minimum_fragrance_support":"Minimum support","attention_median":"Median attention (votes)",
        "rating_median":"Median rating"})[
        ["Classification","Eligible profiles","Minimum support","Median attention (votes)","Median rating"]]

def counts_table(level):
    d=segment_summary[segment_summary.analytical_level==level]
    return d.pivot_table(index="segment",columns="gender_clean",values="profile_count",aggfunc="sum",fill_value=0).reindex(SEGMENTS).reindex(columns=GENDERS).reset_index().rename(columns={"segment":"Segment"})

def metrics_table(level):
    d=segment_summary[segment_summary.analytical_level==level].copy()
    d["segment"]=pd.Categorical(d.segment,categories=SEGMENTS,ordered=True)
    d["gender_clean"]=pd.Categorical(d.gender_clean,categories=GENDERS,ordered=True)
    d=d.sort_values(["gender_clean","segment"])
    return d.rename(columns={"gender_clean":"Classification","segment":"Segment","profile_count":"Profiles",
        "fragrance_assignments":"Fragrance assignments","mean_votes_per_profile":"Mean votes",
        "mean_rating_per_profile":"Mean rating","median_fragrances_per_profile":"Median profile frequency"})[
        ["Classification","Segment","Profiles","Fragrance assignments","Mean votes","Mean rating","Median profile frequency"]]

def example_table(level):
    d=examples[examples.analytical_level==level].sort_values(["classification","segment","profile","rank_by_votes"])
    return d.rename(columns={"classification":"Classification","segment":"Segment","profile":"Profile",
        "rank_by_votes":"Rank","fragrance_name":"Fragrance","brand":"Brand","rating":"Rating",
        "rating_count":"Votes","profile_frequency":"Profile frequency"})[
        ["Classification","Segment","Profile","Rank","Fragrance","Brand","Rating","Votes","Profile frequency"]]

t34=thresholds_table("Dominant accords"); t35=counts_table("Dominant accords"); t36=metrics_table("Dominant accords"); t37=example_table("Dominant accords")
t38=thresholds_table("Dyads"); t39=counts_table("Dyads"); t310=metrics_table("Dyads"); t311=example_table("Dyads")
t312=thresholds_table("Triads"); t313=counts_table("Triads"); t314=metrics_table("Triads"); t315=example_table("Triads")

t316=level_comparison.pivot(index="gender_clean",columns="analytical_level",values="eligible_profile_count").reindex(index=GENDERS,columns=LEVELS).reset_index().rename(columns={"gender_clean":"Classification","Dominant accords":"Dominant accord profiles","Dyads":"Dyad profiles","Triads":"Triad profiles"})
t317=level_comparison.pivot(index="gender_clean",columns="analytical_level",values="fragrance_assignments").reindex(index=GENDERS,columns=LEVELS).reset_index().rename(columns={"gender_clean":"Classification","Dominant accords":"Dominant accord assignments","Dyads":"Dyad assignments","Triads":"Triad assignments"})
t318=level_comparison.rename(columns={"gender_clean":"Classification","analytical_level":"Analytical level","eligible_profile_count":"Eligible profiles","mean_votes_per_profile":"Mean votes per profile","mean_rating_per_profile":"Mean rating per profile"})[["Classification","Analytical level","Eligible profiles","Mean votes per profile","Mean rating per profile"]]
t319=segment_comparison.rename(columns={"analytical_level":"Analytical level","segment":"Segment"})

TABLES = [
(3.1,t31,"Catalogue Distribution by Classification","Fragrance counts and shares within the classified catalogue."),
(3.2,t32,"Most Prevalent Accords by Classification","Five most prevalent recorded accords within each classification."),
(3.3,t33,"Accords with the Largest Prevalence Differences","Top ten accords ranked by maximum minus minimum prevalence, in percentage points."),
(3.4,t34,"Dominant Accord Segmentation Benchmarks","Classification-specific medians and minimum support."),
(3.5,t35,"Dominant Accord Profiles by Segment and Classification","Eligible dominant accord profile counts by relative segment."),
(3.6,t36,"Dominant Accord Segment Metrics","Profile counts, assignments and unweighted profile-level metrics."),
(3.7,t37,"Fragrance Examples for Dominant Accord Profiles","Complete top-three candidate review; final manuscript uses selected profiles."),
(3.8,t38,"Dyad Segmentation Benchmarks","Classification-specific medians and minimum support."),
(3.9,t39,"Dyad Profiles by Segment and Classification","Eligible dyad profile counts by relative segment."),
(3.10,t310,"Dyad Segment Metrics","Profile counts, assignments and unweighted profile-level metrics."),
(3.11,t311,"Fragrance Examples for Dyad Profiles","Complete top-three candidate review; final manuscript uses selected profiles."),
(3.12,t312,"Triad Segmentation Benchmarks","Classification-specific medians and minimum support."),
(3.13,t313,"Triad Profiles by Segment and Classification","Eligible triad profile counts by relative segment."),
(3.14,t314,"Triad Segment Metrics","Profile counts, assignments and unweighted profile-level metrics."),
(3.15,t315,"Fragrance Examples for Triad Profiles","Complete top-three candidate review; final manuscript uses selected profiles."),
(3.16,t316,"Eligible Profiles Across Analytical Levels","Profiles meeting the support threshold by classification and level."),
(3.17,t317,"Fragrance Assignments Across Analytical Levels","Assignments within eligible profiles; totals are not unique catalogue counts."),
(3.18,t318,"Mean Attention and Rating Across Analytical Levels","Unweighted means across eligible profiles within each classification and level."),
(3.19,t319,"Segment Distribution Across Analytical Levels","Eligible profile counts by segment, classification and analytical level.")
]
print("Loaded source exports and prepared",len(TABLES),"table datasets.")

## 1. Tables 3.1–3.19 — first visual draft

In [ ]:
# Render all 19 planned tables. The three fragrance example tables are full
# review listings (1,425 rows across all profile groups), not final book tables.
for number, data, title, caption in TABLES:
    print(f"Table {number}: {title}")
    show_table(data,title,caption)

## 2. Figure 3.1 — Top Five Accord Prevalence by Classification

**Caption:** Comparison of the five most prevalent recorded accords in Men, Women and Unisex fragrances. Values are within-classification prevalence percentages.

In [ ]:
# FIGURE 3.1 — TOP FIVE ACCORD PREVALENCE
fig_3_1, axes = plt.subplots(1,3,figsize=(17,8),dpi=180,sharex=True)
for ax, gender in zip(axes,GENDERS):
    d=prevalence[prevalence.gender_clean==gender].nlargest(5,"prevalence_pct").sort_values("prevalence_pct")
    bars=ax.barh(d.accord.str.title(),d.prevalence_pct,color=STYLE[gender],height=.62)
    for bar,v in zip(bars,d.prevalence_pct):
        ax.text(v+1,bar.get_y()+bar.get_height()/2,f"{v:.1f}%",va="center",fontsize=9)
    ax.set_title(gender,loc="left",fontfamily="DejaVu Serif",fontsize=15,fontweight="bold")
    ax.set_xlim(0,100); ax.set_xlabel("Prevalence (%)"); ax.grid(axis="x",color=STYLE["grid"]); ax.set_axisbelow(True)
    ax.tick_params(length=0,labelsize=9)
    [sp.set_visible(False) for sp in ax.spines.values()]
fig_3_1.text(.06,.97,"FRAGRANCE INTELLIGENCE  /  CHAPTER 3",color=STYLE["accent"],fontsize=9,fontweight="bold")
fig_3_1.text(.06,.93,"Top Five Accord Prevalence by Classification",fontfamily="DejaVu Serif",fontsize=21,fontweight="bold")
fig_3_1.text(.06,.885,"Percentage of fragrances within each classification containing each accord.",color=STYLE["muted"],fontsize=12)
fig_3_1.subplots_adjust(left=.08,right=.98,top=.78,bottom=.12,wspace=.35)
display(fig_3_1); plt.close(fig_3_1)

## 3. Figure 3.2 — Accord Prevalence Across Gender Classifications

**Caption:** Heatmap of within-classification prevalence for the 30 accords with the largest prevalence spread across the three classifications.

In [ ]:
# FIGURE 3.2 — ACCORD PREVALENCE HEATMAP
hw=prevalence.pivot(index="accord",columns="gender_clean",values="prevalence_pct").reindex(columns=GENDERS)
hw["spread"]=hw.max(axis=1)-hw.min(axis=1)
hw=hw.nlargest(30,"spread")[GENDERS]
fig_3_2,ax=plt.subplots(figsize=(10,12),dpi=180)
cmap=LinearSegmentedColormap.from_list("ivory_bronze",[STYLE["bg"],"#D8C3AD",STYLE["accent"]])
im=ax.imshow(hw.values,aspect="auto",cmap=cmap,vmin=0,vmax=100)
ax.set_yticks(np.arange(len(hw))); ax.set_yticklabels(hw.index.str.title(),fontsize=9)
ax.set_xticks(np.arange(3)); ax.set_xticklabels(GENDERS,fontsize=10)
for i in range(len(hw)):
    for j in range(3):
        ax.text(j,i,f"{hw.iloc[i,j]:.1f}%",ha="center",va="center",fontsize=8,color=STYLE["text"])
ax.tick_params(length=0)
[sp.set_visible(False) for sp in ax.spines.values()]
cb=fig_3_2.colorbar(im,ax=ax,fraction=.025,pad=.02); cb.set_label("Prevalence (%)"); cb.outline.set_visible(False)
fig_3_2.text(.06,.97,"FRAGRANCE INTELLIGENCE  /  CHAPTER 3",color=STYLE["accent"],fontsize=9,fontweight="bold")
fig_3_2.text(.06,.93,"Accord Prevalence Across Gender Classifications",fontfamily="DejaVu Serif",fontsize=20,fontweight="bold")
fig_3_2.text(.06,.885,"Top 30 accords ranked by maximum prevalence spread.",color=STYLE["muted"],fontsize=12)
fig_3_2.subplots_adjust(left=.24,right=.91,top=.90,bottom=.06)
display(fig_3_2); plt.close(fig_3_2)

## 4. Figure 3.3 — Dominant Accord Attention and Rating Segmentation

**Caption:** Eligible dominant accord profiles plotted by mean votes per fragrance and mean rating. Colour indicates the relative segment; dashed lines show the classification-specific medians.

In [ ]:
# FIGURE 3.3 — DOMINANT ACCORD SEGMENTATION
FIGURE_OBJECTS_CH3 = []

def plot_level_scatter(data, title, fig_num):
    fig,axes=plt.subplots(1,3,figsize=(17,7.5),dpi=180,sharey=True)
    for ax,gender in zip(axes,GENDERS):
        d=data[data.gender_clean==gender]
        xm=d.mean_votes.median(); ym=d.mean_rating.median()
        for seg in SEGMENTS:
            p=d[d.segment.astype(str)==seg]
            ax.scatter(p.mean_votes,p.mean_rating,s=40,color=SEGMENT_COLORS[seg],
                       alpha=.78,edgecolors=STYLE["bg"],linewidths=.5,label=seg)
        ax.axvline(xm,color=STYLE["muted"],ls="--",lw=1); ax.axhline(ym,color=STYLE["muted"],ls="--",lw=1)
        ax.set_title(gender,loc="left",fontfamily="DejaVu Serif",fontsize=15,fontweight="bold")
        ax.set_xlabel("Mean votes per fragrance"); ax.grid(color=STYLE["grid"],lw=.7); ax.set_axisbelow(True)
        ax.tick_params(length=0,labelsize=9); [sp.set_visible(False) for sp in ax.spines.values()]
    axes[0].set_ylabel("Mean fragrance rating")
    handles,labels=axes[0].get_legend_handles_labels()
    fig.legend(handles,labels,loc="lower center",bbox_to_anchor=(.5,.02),ncol=4,frameon=False,fontsize=9)
    fig.text(.06,.97,"FRAGRANCE INTELLIGENCE  /  CHAPTER 3",color=STYLE["accent"],fontsize=9,fontweight="bold")
    fig.text(.06,.93,title,fontfamily="DejaVu Serif",fontsize=20,fontweight="bold")
    fig.text(.06,.885,"Classification-specific medians define the four relative segments.",color=STYLE["muted"],fontsize=12)
    fig.subplots_adjust(left=.08,right=.98,top=.78,bottom=.20,wspace=.25)
    FIGURE_OBJECTS_CH3.append((fig, f"figure_3_{fig_num}_segmentation"))
    display(fig)

plot_level_scatter(dominant,"Dominant Accord Profiles — Attention and Rating",3)

## 5. Figure 3.4 — Dyad Attention and Rating Segmentation

**Caption:** Eligible ordered dyad profiles plotted by mean votes per fragrance and mean rating. Colour indicates the relative segment; dashed lines show the classification-specific medians.

In [ ]:
plot_level_scatter(dyads,"Dyad Profiles — Attention and Rating",4)

## 6. Figure 3.5 — Triad Attention and Rating Segmentation

**Caption:** Eligible ordered triad profiles plotted by mean votes per fragrance and mean rating. Colour indicates the relative segment; dashed lines show the classification-specific medians.

In [ ]:
plot_level_scatter(triads,"Triad Profiles — Attention and Rating",5)

## 7. Figure 3.6 — Cross-Level Comparison of Profile Segmentation

**Caption:** Number of eligible profiles in each median-based segment, compared across dominant accords, dyads and triads within each classification.

In [ ]:
# FIGURE 3.6 — CROSS-LEVEL SEGMENT DISTRIBUTION
plot=segment_summary.copy()
fig_3_6,axes=plt.subplots(1,3,figsize=(17,8),dpi=180,sharey=True)
for ax,gender in zip(axes,GENDERS):
    d=plot[plot.gender_clean==gender]
    p=d.pivot_table(index="analytical_level",columns="segment",values="profile_count",aggfunc="sum",fill_value=0)
    p=p.reindex(index=LEVELS,columns=SEGMENTS,fill_value=0)
    bottom=np.zeros(len(p))
    for seg in SEGMENTS:
        vals=p[seg].values
        ax.bar([str(x) for x in p.index],vals,bottom=bottom,color=SEGMENT_COLORS[seg],label=seg,width=.62)
        bottom+=vals
    ax.set_title(gender,loc="left",fontfamily="DejaVu Serif",fontsize=15,fontweight="bold")
    ax.tick_params(axis="x",rotation=18,length=0,labelsize=9); ax.tick_params(axis="y",length=0)
    ax.grid(axis="y",color=STYLE["grid"]); ax.set_axisbelow(True)
    [sp.set_visible(False) for sp in ax.spines.values()]
axes[0].set_ylabel("Eligible profile count")
handles,labels=axes[0].get_legend_handles_labels()
fig_3_6.legend(handles,labels,loc="lower center",bbox_to_anchor=(.5,.02),ncol=4,frameon=False,fontsize=9)
fig_3_6.text(.06,.97,"FRAGRANCE INTELLIGENCE  /  CHAPTER 3",color=STYLE["accent"],fontsize=9,fontweight="bold")
fig_3_6.text(.06,.93,"Profile Segmentation Across Analytical Levels",fontfamily="DejaVu Serif",fontsize=20,fontweight="bold")
fig_3_6.text(.06,.885,"Stacked eligible profile counts by segment, analytical level and classification.",color=STYLE["muted"],fontsize=12)
fig_3_6.subplots_adjust(left=.08,right=.98,top=.78,bottom=.20,wspace=.28)
FIGURE_OBJECTS_CH3.append((fig_3_6,"figure_3_6_cross_level_segmentation"))
display(fig_3_6)

## 8. Export draft figures — PNG and SVG

Run after reviewing the rendered figures. This export cell saves the six figures to the shared visualisations volume. Table exports are intentionally deferred until the table design has been reviewed.

In [ ]:
# Export all six Chapter 3 figures as high-resolution PNG and SVG masters.
# Run after all six figure cells have been executed.
FIGURES_CH3 = [
    (fig_3_1,"figure_3_1_top_five_accord_prevalence"),
    (fig_3_2,"figure_3_2_accord_prevalence_heatmap"),
    *FIGURE_OBJECTS_CH3
]
assert len(FIGURES_CH3) == 6, f"Expected 6 figures, found {len(FIGURES_CH3)}"
saved=[]
for fig,stem in FIGURES_CH3:
    for ext,kwargs in [("png",{"dpi":400,"bbox_inches":"tight"}),("svg",{"bbox_inches":"tight"})]:
        path=OUTPUT_DIR/f"{stem}.{ext}"
        fig.savefig(path,format=ext,facecolor=STYLE["bg"],**kwargs)
        assert path.is_file() and path.stat().st_size>0, f"Export failed: {path}"
        saved.append(path)
print("Verified figure exports:",len(saved))
for p in saved: print(p)

## 9. Review checklist

- [ ] Review all 19 tables for hierarchy, spacing, column widths, alignment, precision and long labels.
- [ ] Review the three complete fragrance-example candidate listings; select profiles for the manuscript later.
- [ ] Review all six figures for readability, label density, ordering and colour.
- [ ] Decide which table layouts should become reusable templates for the whole book.
- [ ] After the design is agreed, update the shared table and figure standards in Notebook 14 and apply them across Chapters 1–3.